# Experiment 11 — Rao (2021) reproduction, corrected comparison protocol

This notebook reproduces the Rao (2021) workflow as closely as the paper permits:

- BraTS 2021: **577 patients**
- FLAIR only
- whole-tumor mask
- 93 PyRadiomics features
- global radiomics
- local radiomics from **3×3×3** neighborhoods
- tumor crop resized to **33×33×33**
- activation proportion `%A`
- feature-wise Bayesian classifier + Gaussian KDE
- 10-fold stratified cross-validation

## Corrections applied

1. One shared patient-fold assignment is created/reused by Experiments 11–13.
2. KDE bandwidth tuning uses explicit 5-fold inner CV instead of an undocumented Leave-One-Out assumption.
3. The t-test variance assumption is exposed as a parameter.
4. The exact published names of the 46 global and 62 local significant features are checked.
5. PyRadiomics settings/version are made explicit; unpublished values are marked as assumptions.
6. Feature-map reading is vectorized.
7. Cache reuse requires a matching configuration hash.
8. Resampling preserves physical extent.

> **33×33×33 is the resized tumor crop. 3×3×3 is the local radiomics neighborhood.**
> They are different stages and are not interchangeable.


## 1. Install dependencies


In [ ]:
# Run only if needed
# !pip install numpy pandas scipy SimpleITK pyradiomics scikit-learn matplotlib statsmodels


## 2. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import SimpleITK as sitk
import radiomics
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.stats import ttest_ind

from radiomics import featureextractor

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KernelDensity
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)


## 3. Published experiment constants

The 93 features correspond exactly to the non-shape PyRadiomics groups:

- First-order: 18
- GLCM: 24
- GLRLM: 16
- GLSZM: 16
- GLDM: 14
- NGTDM: 5

Total:

\[
18+24+16+16+14+5 = 93
\]

The paper uses whole tumor = all provided tumor regions combined.


In [ ]:
RANDOM_STATE = 42
N_PATIENTS_EXPECTED = 577
EXPECTED_MGMT_1 = 301
EXPECTED_MGMT_0 = 276

LOCAL_CROP_SIZE = (33, 33, 33)
LOCAL_KERNEL_RADIUS = 1
LOCAL_MIN_TUMOR_VOXELS = 1

N_SPLITS = 10
P_VALUE_THRESHOLD = 0.05

# Rao only states "t-tests"; the equal-variance assumption is not reported.
TTEST_EQUAL_VAR = True

# Rao states GridSearchCV over a manually selected bandwidth domain,
# but does not publish the exact inner-CV count or grid.
KDE_INNER_CV = 5
KDE_BANDWIDTH_GRID = np.logspace(-2, 1, 40)

FEATURE_CLASSES = [
    "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm",
]

RAO_GLOBAL_46 = ['glcm_Correlation', 'glcm_Imc2', 'glcm_DifferenceVariance', 'glcm_Idm', 'glcm_Contrast', 'glcm_Id', 'firstorder_RobustMeanAbsoluteDeviation', 'firstorder_InterquartileRange', 'glcm_Autocorrelation', 'firstorder_MeanAbsoluteDeviation', 'firstorder_Minimum', 'glcm_DifferenceAverage', 'glcm_DifferenceEntropy', 'glcm_JointAverage', 'firstorder_Entropy', 'firstorder_Range', 'glcm_InverseVariance', 'firstorder_10Percentile', 'firstorder_Maximum', 'firstorder_Median', 'firstorder_Mean', 'firstorder_90Percentile', 'firstorder_RootMeanSquared', 'glszm_GrayLevelNonUniformityNormalized', 'glszm_ZoneEntropy', 'glszm_SmallAreaEmphasis', 'glszm_SmallAreaLowGrayLevelEmphasis', 'glrlm_ShortRunHighGrayLevelEmphasis', 'glszm_SmallAreaHighGrayLevelEmphasis', 'glrlm_HighGrayLevelRunEmphasis', 'glcm_SumEntropy', 'glrlm_RunEntropy', 'gldm_DependenceNonUniformityNormalized', 'gldm_SmallDependenceEmphasis', 'glszm_HighGrayLevelZoneEmphasis', 'glszm_SizeZoneNonUniformityNormalized', 'ngtdm_Complexity', 'glcm_SumAverage', 'gldm_SmallDependenceHighGrayLevelEmphasis', 'glszm_ZonePercentage', 'glcm_JointEntropy', 'gldm_DependenceEntropy', 'gldm_LargeDependenceHighGrayLevelEmphasis', 'gldm_HighGrayLevelEmphasis', 'glcm_MCC', 'glrlm_LongRunHighGrayLevelEmphasis']
RAO_LOCAL_62 = ['glcm_DifferenceEntropy', 'firstorder_Skewness', 'firstorder_Minimum', 'firstorder_Range', 'glcm_Imc1', 'firstorder_Kurtosis', 'glcm_JointAverage', 'glcm_SumAverage', 'glcm_Autocorrelation', 'firstorder_Entropy', 'glcm_DifferenceVariance', 'firstorder_Uniformity', 'glcm_ClusterProminence', 'glcm_ClusterTendency', 'firstorder_Variance', 'firstorder_MeanAbsoluteDeviation', 'firstorder_RobustMeanAbsoluteDeviation', 'firstorder_InterquartileRange', 'glcm_Imc2', 'glcm_MCC', 'firstorder_Maximum', 'glcm_Idm', 'glcm_Id', 'glcm_ClusterShade', 'glcm_Contrast', 'glcm_InverseVariance', 'glcm_DifferenceAverage', 'firstorder_90Percentile', 'firstorder_Median', 'firstorder_RootMeanSquared', 'firstorder_Mean', 'glszm_HighGrayLevelZoneEmphasis', 'glrlm_LongRunEmphasis', 'gldm_SmallDependenceHighGrayLevelEmphasis', 'glrlm_GrayLevelNonUniformityNormalized', 'glszm_SmallAreaHighGrayLevelEmphasis', 'gldm_GrayLevelVariance', 'ngtdm_Complexity', 'ngtdm_Strength', 'glrlm_RunPercentage', 'gldm_DependenceNonUniformityNormalized', 'glszm_LargeAreaLowGrayLevelEmphasis', 'glrlm_LongRunHighGrayLevelEmphasis', 'glrlm_RunVariance', 'glrlm_ShortRunHighGrayLevelEmphasis', 'glrlm_RunLengthNonUniformityNormalized', 'gldm_LargeDependenceEmphasis', 'glrlm_GrayLevelVariance', 'gldm_DependenceVariance', 'glszm_ZonePercentage', 'ngtdm_Contrast', 'gldm_HighGrayLevelEmphasis', 'glszm_SmallAreaEmphasis', 'glszm_SizeZoneNonUniformityNormalized', 'glrlm_ShortRunEmphasis', 'glszm_GrayLevelNonUniformityNormalized', 'ngtdm_Busyness', 'glszm_GrayLevelVariance', 'glcm_SumSquares', 'gldm_SmallDependenceEmphasis', 'glrlm_HighGrayLevelRunEmphasis', 'gldm_LargeDependenceLowGrayLevelEmphasis']

assert len(RAO_GLOBAL_46) == 46
assert len(RAO_LOCAL_62) == 62


## 4. Input manifest

Prepare one CSV with:

```text
patient_id,flair_path,seg_path,mgmt
BraTS2021_00000,/path/..._flair.nii.gz,/path/..._seg.nii.gz,1
...
```

Where:
- `mgmt = 1` → methylated
- `mgmt = 0` → unmethylated
- `seg != 0` → whole tumor

This matches the paper's merging of Task 1 segmentation data and Task 2 MGMT labels by unique patient ID.


In [ ]:
MANIFEST_PATH = Path("/path/to/brats2021_577_manifest.csv")  # <-- change

OUTDIR = Path("outputs_rao2021")
OUTDIR.mkdir(parents=True, exist_ok=True)

SHARED_DIR = Path("shared_rao_comparison")
SHARED_DIR.mkdir(parents=True, exist_ok=True)
SHARED_FOLD_CSV = SHARED_DIR / "mgmt_577_fold_assignment_10fold.csv"

REUSE_GLOBAL_FEATURES = True
REUSE_LOCAL_PERCENT_ACTIVATION = True

FLAIR_INTERPOLATOR = sitk.sitkLinear
MASK_INTERPOLATOR = sitk.sitkNearestNeighbor

# Complete low-level settings are not published by Rao.
# These are explicit implementation assumptions/defaults.
PYRAD_SETTINGS = {
    "binWidth": 25,
    "normalize": False,
    "resampledPixelSpacing": None,
    "force2D": False,
    "distances": [1],
}

print("Installed PyRadiomics:", radiomics.__version__)
print("PyRadiomics settings:", PYRAD_SETTINGS)


## 5. Load and verify the 577-patient cohort

In [ ]:
def load_manifest(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"patient_id", "flair_path", "seg_path", "mgmt"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing manifest columns: {sorted(missing)}")

    df["patient_id"] = df["patient_id"].astype(str)
    df["mgmt"] = df["mgmt"].astype(int)
    df = df.sort_values("patient_id").reset_index(drop=True)

    if df["patient_id"].duplicated().any():
        raise ValueError("Duplicate patient_id values found.")
    if len(df) != N_PATIENTS_EXPECTED:
        raise ValueError(f"Expected exactly 577 patients, found {len(df)}.")

    counts = df["mgmt"].value_counts().to_dict()
    if counts.get(1, 0) != EXPECTED_MGMT_1 or counts.get(0, 0) != EXPECTED_MGMT_0:
        raise ValueError(
            f"Expected MGMT counts 1={EXPECTED_MGMT_1}, 0={EXPECTED_MGMT_0}; found {counts}."
        )

    for c in ["flair_path", "seg_path"]:
        df[c] = df[c].map(lambda x: str(Path(x).expanduser().resolve()))
        bad = [p for p in df[c] if not Path(p).exists()]
        if bad:
            raise FileNotFoundError(f"{c}: missing file, example: {bad[0]}")
    return df


def get_or_create_shared_folds(cohort: pd.DataFrame) -> pd.DataFrame:
    expected = cohort[["patient_id", "mgmt"]].sort_values("patient_id").reset_index(drop=True)

    if SHARED_FOLD_CSV.exists():
        folds = pd.read_csv(SHARED_FOLD_CSV)
        folds["patient_id"] = folds["patient_id"].astype(str)
        required = {"patient_id", "mgmt", "fold"}
        if required - set(folds.columns):
            raise ValueError("Shared fold file has an invalid schema.")

        merged = expected.merge(
            folds[["patient_id", "mgmt", "fold"]],
            on=["patient_id", "mgmt"],
            how="outer",
            indicator=True,
        )
        if len(folds) != 577 or not (merged["_merge"] == "both").all():
            raise ValueError("Existing shared folds do not match the current 577-patient cohort.")
        return folds.sort_values("patient_id").reset_index(drop=True)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    y = expected["mgmt"].to_numpy()

    for fold, (_, va_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        for idx in va_idx:
            rows.append({
                "patient_id": expected.iloc[idx]["patient_id"],
                "mgmt": int(expected.iloc[idx]["mgmt"]),
                "fold": fold,
            })

    folds = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    folds.to_csv(SHARED_FOLD_CSV, index=False)
    return folds


def stable_hash(payload) -> str:
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def cohort_signature(df: pd.DataFrame):
    return [
        {
            "patient_id": str(r.patient_id),
            "mgmt": int(r.mgmt),
            "flair_path": str(r.flair_path),
            "seg_path": str(r.seg_path),
        }
        for r in df.sort_values("patient_id").itertuples(index=False)
    ]


manifest = load_manifest(MANIFEST_PATH)
shared_folds = get_or_create_shared_folds(manifest)

print("Patients:", len(manifest))
print(manifest["mgmt"].value_counts().sort_index())
print(shared_folds.groupby(["fold", "mgmt"]).size().unstack(fill_value=0))
display(manifest.head())


## 6. Build the 93-feature PyRadiomics extractor

Shape features are intentionally excluded because the paper describes 93 first/second-order textural features.


In [ ]:
def build_93_extractor(voxel_based=False):
    extractor = featureextractor.RadiomicsFeatureExtractor(**PYRAD_SETTINGS)
    extractor.disableAllFeatures()
    for cls in FEATURE_CLASSES:
        extractor.enableFeatureClassByName(cls)
    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName("Original")

    if voxel_based:
        extractor.settings["kernelRadius"] = LOCAL_KERNEL_RADIUS
        extractor.settings["maskedKernel"] = False
    return extractor


def radiomics_feature_names_from_result(result):
    return [k.replace("original_", "") for k in result if k.startswith("original_")]


## 7. Global radiomics — whole tumor

For each patient:
1. use FLAIR;
2. use whole-tumor mask (`seg != 0`);
3. extract 93 features;
4. obtain one 93-length vector.

The paper then Z-normalizes each feature using the cohort mean and standard deviation.


In [ ]:
def make_whole_tumor_mask(seg_img: sitk.Image) -> sitk.Image:
    arr = sitk.GetArrayFromImage(seg_img)
    mask_arr = (arr != 0).astype(np.uint8)
    mask = sitk.GetImageFromArray(mask_arr)
    mask.CopyInformation(seg_img)
    return mask


def extract_global_features(manifest_df: pd.DataFrame) -> pd.DataFrame:
    out_path = OUTDIR / "global_93_raw.csv"
    meta_path = OUTDIR / "global_93_raw.meta.json"

    payload = {
        "stage": "global_93",
        "cohort": cohort_signature(manifest_df),
        "pyradiomics_version": radiomics.__version__,
        "pyrad_settings": PYRAD_SETTINGS,
        "feature_classes": FEATURE_CLASSES,
    }
    h = stable_hash(payload)

    if REUSE_GLOBAL_FEATURES and out_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("config_hash") == h:
            print("Reusing validated cache:", out_path)
            return pd.read_csv(out_path)
        print("Global cache hash mismatch; recomputing.")

    extractor = build_93_extractor(voxel_based=False)
    rows = []

    for i, row in manifest_df.iterrows():
        print(f"[Global {i+1}/{len(manifest_df)}] {row['patient_id']}")
        flair = sitk.ReadImage(row["flair_path"])
        seg = sitk.ReadImage(row["seg_path"])
        result = extractor.execute(flair, make_whole_tumor_mask(seg))

        record = {"patient_id": row["patient_id"], "mgmt": int(row["mgmt"])}
        for k, v in result.items():
            if k.startswith("original_"):
                try:
                    record[k.replace("original_", "")] = float(v)
                except Exception:
                    record[k.replace("original_", "")] = np.nan
        rows.append(record)

    df = pd.DataFrame(rows)
    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]
    if len(feature_cols) != 93:
        raise RuntimeError(f"Expected 93 features, found {len(feature_cols)}.")

    df.to_csv(out_path, index=False)
    meta_path.write_text(
        json.dumps({"config_hash": h, "payload": payload}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return df


global_raw = extract_global_features(manifest)
display(global_raw.head())


## 8. Global Z-normalization and t-test feature selection

The paper describes:
- Z-normalizing a patient's raw feature value using the mean and SD of that feature across all patients;
- performing a t-test between MGMT 1 and MGMT 0;
- selecting `p < 0.05`;
- obtaining **46/93 significant global features**.

### Methodological note

This is reproduced as described. Under stricter modern ML practice, normalization and feature selection would normally be fit within each training fold to avoid leakage. This notebook preserves the paper workflow for comparison.


In [ ]:
def z_normalize_cohort(df: pd.DataFrame):
    meta = {"patient_id", "mgmt"}
    feature_cols = [c for c in df.columns if c not in meta]
    z = df.copy()
    stats = {}

    for c in feature_cols:
        mu = df[c].mean()
        sd = df[c].std(ddof=0)
        stats[c] = {"mean": float(mu), "sd": float(sd)}
        z[c] = (df[c] - mu) / sd if sd > 0 and np.isfinite(sd) else 0.0
    return z, stats


def ttest_select(df: pd.DataFrame, p_threshold=0.05):
    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]
    rows = []
    for c in feature_cols:
        x1 = df.loc[df["mgmt"] == 1, c].dropna().to_numpy()
        x0 = df.loc[df["mgmt"] == 0, c].dropna().to_numpy()
        stat, p = ttest_ind(
            x1, x0,
            equal_var=TTEST_EQUAL_VAR,
            nan_policy="omit",
        )
        rows.append({
            "feature": c,
            "t_stat": stat,
            "p_value": p,
            "significant": bool(np.isfinite(p) and p < p_threshold),
        })
    return pd.DataFrame(rows).sort_values("p_value")


def compare_feature_names(observed, expected, label):
    obs, exp = set(observed), set(expected)
    report = {
        "set": label,
        "observed_count": len(obs),
        "expected_count": len(exp),
        "exact_match": obs == exp,
        "missing_from_reproduction": sorted(exp - obs),
        "extra_in_reproduction": sorted(obs - exp),
    }
    print(json.dumps(report, indent=2, ensure_ascii=False))
    return report


global_z, global_z_stats = z_normalize_cohort(global_raw)
global_ttest = ttest_select(global_z, P_VALUE_THRESHOLD)
global_sig = global_ttest[global_ttest["significant"]]["feature"].tolist()
global_feature_match = compare_feature_names(global_sig, RAO_GLOBAL_46, "global")

global_z.to_csv(OUTDIR / "global_93_z_normalized.csv", index=False)
global_ttest.to_csv(OUTDIR / "global_ttest_results.csv", index=False)


## 9. Crop whole-tumor region and resize to 33×33×33

The paper forms a minimum-size cube around the general tumor region and resizes it to a uniform **33×33×33** voxels.

Implementation here:
- determine the tight whole-tumor bounding box;
- expand shorter dimensions symmetrically to create a cube;
- crop FLAIR and segmentation;
- resample both to 33×33×33;
- FLAIR: linear interpolation;
- segmentation: nearest-neighbor interpolation.

The paper does not publish the exact interpolation choices, so these are explicit assumptions.


In [ ]:
def bbox_from_mask(mask_zyx):
    coords = np.argwhere(mask_zyx)
    if coords.size == 0:
        raise ValueError("Empty tumor mask.")
    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1
    return (x0, y0, z0), (x1, y1, z1)


def cube_bounds(mask_zyx):
    (x0, y0, z0), (x1, y1, z1) = bbox_from_mask(mask_zyx)
    side = max(x1-x0, y1-y0, z1-z0)
    cx, cy, cz = (x0+x1-1)/2, (y0+y1-1)/2, (z0+z1-1)/2

    starts = [
        int(round(cx-(side-1)/2)),
        int(round(cy-(side-1)/2)),
        int(round(cz-(side-1)/2)),
    ]
    dims = [mask_zyx.shape[2], mask_zyx.shape[1], mask_zyx.shape[0]]

    for d in range(3):
        starts[d] = max(0, starts[d])
        if starts[d] + side > dims[d]:
            starts[d] = max(0, dims[d] - side)

    size = [min(side, dims[d]-starts[d]) for d in range(3)]
    return tuple(starts), tuple(size)


def crop_sitk(image, start_xyz, size_xyz):
    return sitk.RegionOfInterest(
        image,
        size=[int(v) for v in size_xyz],
        index=[int(v) for v in start_xyz],
    )


def resample_to_33(image, is_mask=False):
    # Rao: standardized tumor crop is 33×33×33.
    # The local radiomics neighborhood is separately 3×3×3.
    old_size = np.array(image.GetSize(), dtype=float)
    old_spacing = np.array(image.GetSpacing(), dtype=float)
    new_size = np.array([33, 33, 33], dtype=int)

    new_spacing = old_spacing * np.maximum(old_size - 1, 1) / np.maximum(new_size - 1, 1)

    r = sitk.ResampleImageFilter()
    r.SetSize([int(v) for v in new_size])
    r.SetOutputSpacing([float(v) for v in new_spacing])
    r.SetOutputOrigin(image.GetOrigin())
    r.SetOutputDirection(image.GetDirection())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(MASK_INTERPOLATOR if is_mask else FLAIR_INTERPOLATOR)

    pixel_type = sitk.sitkUInt8 if is_mask else sitk.sitkFloat32
    return r.Execute(sitk.Cast(image, pixel_type))


def prepare_local_33(flair_path, seg_path):
    flair = sitk.ReadImage(str(flair_path))
    seg = sitk.ReadImage(str(seg_path))

    tumor = sitk.GetArrayFromImage(seg) != 0
    start, size = cube_bounds(tumor)

    flair_crop = crop_sitk(flair, start, size)
    seg_crop = crop_sitk(seg, start, size)

    flair_33 = resample_to_33(flair_crop, is_mask=False)

    mask_arr = (sitk.GetArrayFromImage(seg_crop) != 0).astype(np.uint8)
    mask = sitk.GetImageFromArray(mask_arr)
    mask.CopyInformation(seg_crop)
    tumor_33 = resample_to_33(mask, is_mask=True)

    return flair_33, tumor_33


## 10. Centers eligible for local 3×3×3 radiomics

A local 3×3×3 cube is processed when it contains at least **1 tumor voxel out of 27**.

We identify all eligible center voxels using a 3×3×3 convolution over the resized tumor mask.


In [ ]:
def eligible_center_mask(tumor_33: sitk.Image) -> sitk.Image:
    arr = sitk.GetArrayFromImage(tumor_33).astype(np.uint8)

    local_count = ndi.convolve(
        arr,
        np.ones((3, 3, 3), dtype=np.uint8),
        mode="constant",
        cval=0,
    )

    eligible = (local_count >= LOCAL_MIN_TUMOR_VOXELS).astype(np.uint8)

    # Exclude border centers because a full 3x3x3 cube does not exist there.
    eligible[[0, -1], :, :] = 0
    eligible[:, [0, -1], :] = 0
    eligible[:, :, [0, -1]] = 0

    mask = sitk.GetImageFromArray(eligible)
    mask.CopyInformation(tumor_33)
    return mask


## 11. Local 93-feature value maps

Instead of calling PyRadiomics separately tens of thousands of times per patient, this notebook uses PyRadiomics' `voxelBased=True` mode with `kernelRadius=1`.

This computes the same *type* of local 3×3×3 radiomic neighborhood much more efficiently.

### Reproducibility qualification

The paper describes a literal sliding 3×3×3 cube implementation but does not publish source code. PyRadiomics voxel-based extraction is therefore a practical implementation approximation, while preserving:
- the same PyRadiomics feature definitions;
- local 3×3×3 neighborhoods;
- the same tumor-containing-cube eligibility rule.


In [ ]:
def extract_local_feature_maps(flair_33, tumor_33):
    center_mask = eligible_center_mask(tumor_33)

    extractor = build_93_extractor(voxel_based=True)

    result = extractor.execute(
        flair_33,
        center_mask,
        label=1,
        voxelBased=True,
    )

    maps = {}
    for k, v in result.items():
        if k.startswith("original_") and isinstance(v, sitk.Image):
            maps[k.replace("original_", "")] = v

    return maps, center_mask


## 12. Activation maps and `%A`

For every feature, Rao defines a local cube as:
- **activated** if local feature value ≥ global median value for that feature across patients;
- **non-activated** otherwise.

For each patient:

\[
\%A = \frac{A}{A+N}
\]

where `A` is the number of activated local positions and `N` is the number of non-activated local positions.

The global medians are calculated from the global 93-feature cohort table, as described in the paper.


In [ ]:
def feature_global_medians(global_df):
    return {
        c: float(global_df[c].median())
        for c in global_df.columns
        if c not in {"patient_id", "mgmt"}
    }


GLOBAL_MEDIANS = feature_global_medians(global_raw)


def get_map_values_at_mask(feature_img, mask_img):
    aligned_mask = sitk.Resample(
        mask_img,
        feature_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )
    f = sitk.GetArrayFromImage(feature_img)
    m = sitk.GetArrayFromImage(aligned_mask).astype(bool)
    return np.asarray(f[m], dtype=float)


def extract_local_percent_activation(manifest_df, global_medians):
    out_path = OUTDIR / "local_93_percent_activation.csv"
    meta_path = OUTDIR / "local_93_percent_activation.meta.json"

    payload = {
        "stage": "local_93_activation",
        "cohort": cohort_signature(manifest_df),
        "pyradiomics_version": radiomics.__version__,
        "pyrad_settings": PYRAD_SETTINGS,
        "crop_size": [33, 33, 33],
        "local_kernel_radius": LOCAL_KERNEL_RADIUS,
        "local_neighborhood": "3x3x3",
        "min_tumor_voxels": LOCAL_MIN_TUMOR_VOXELS,
    }
    h = stable_hash(payload)

    if REUSE_LOCAL_PERCENT_ACTIVATION and out_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("config_hash") == h:
            print("Reusing validated cache:", out_path)
            df = pd.read_csv(out_path)
            if len(df) != 577:
                raise RuntimeError("Cached local features do not contain 577 patients.")
            return df
        print("Local cache hash mismatch; recomputing.")

    rows = []
    for i, row in manifest_df.iterrows():
        pid = row["patient_id"]
        print(f"[Local {i+1}/{len(manifest_df)}] {pid}")

        flair_33, tumor_33 = prepare_local_33(row["flair_path"], row["seg_path"])
        fmap_dict, center_mask = extract_local_feature_maps(flair_33, tumor_33)

        record = {"patient_id": pid, "mgmt": int(row["mgmt"])}

        for feat, fmap in fmap_dict.items():
            values = get_map_values_at_mask(fmap, center_mask)
            values = values[np.isfinite(values)]
            if values.size == 0 or feat not in global_medians:
                record[feat] = np.nan
            else:
                record[feat] = float(np.mean(values >= global_medians[feat]))
        rows.append(record)

    df = pd.DataFrame(rows)
    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]

    if len(df) != 577:
        raise RuntimeError(f"Expected 577 patients, found {len(df)}.")
    if len(feature_cols) != 93:
        raise RuntimeError(f"Expected 93 local features, found {len(feature_cols)}.")

    df.to_csv(out_path, index=False)
    meta_path.write_text(
        json.dumps({"config_hash": h, "payload": payload}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return df


local_activation = extract_local_percent_activation(manifest, GLOBAL_MEDIANS)
display(local_activation.head())


## 13. Local t-test feature selection

The paper reports:
- **62/93** local `%A` features significant at `p < 0.05`.

This cell reproduces that test.


In [ ]:
local_ttest = ttest_select(local_activation, P_VALUE_THRESHOLD)
local_sig = local_ttest[local_ttest["significant"]]["feature"].tolist()
local_feature_match = compare_feature_names(local_sig, RAO_LOCAL_62, "local")

local_ttest.to_csv(OUTDIR / "local_ttest_results.csv", index=False)

(OUTDIR / "rao_feature_name_match.json").write_text(
    json.dumps(
        {"global": global_feature_match, "local": local_feature_match},
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


## 14. KDE Bayesian classifier

For a single feature `x`, the classifier compares:

\[
p(x|MGMT=1)P(MGMT=1)
\]

with

\[
p(x|MGMT=0)P(MGMT=0)
\]

Two separate Gaussian KDE models are fit:
- one for MGMT 1;
- one for MGMT 0.

The KDE bandwidth is optimized inside each training fold using `GridSearchCV`, matching the paper's stated method.

### Unspecified paper detail

The exact manually-set bandwidth search domain is not reported. This notebook uses `KDE_BANDWIDTH_GRID` defined above.


In [ ]:
def fit_best_kde_1d(values):
    values = np.asarray(values, dtype=float).reshape(-1, 1)

    if len(values) < KDE_INNER_CV:
        raise ValueError("Not enough class-specific samples for KDE inner CV.")

    search = GridSearchCV(
        KernelDensity(kernel="gaussian"),
        {"bandwidth": KDE_BANDWIDTH_GRID},
        cv=KDE_INNER_CV,
        n_jobs=-1,
    )
    search.fit(values)

    return search.best_estimator_, float(search.best_params_["bandwidth"])


def bayes_kde_predict(train_x, train_y, test_x):
    train_x = np.asarray(train_x, dtype=float)
    train_y = np.asarray(train_y, dtype=int)
    test_x = np.asarray(test_x, dtype=float)

    x1 = train_x[train_y == 1]
    x0 = train_x[train_y == 0]

    kde1, bw1 = fit_best_kde_1d(x1)
    kde0, bw0 = fit_best_kde_1d(x0)

    p1 = len(x1) / len(train_y)
    p0 = len(x0) / len(train_y)

    log1 = kde1.score_samples(test_x.reshape(-1, 1)) + np.log(p1)
    log0 = kde0.score_samples(test_x.reshape(-1, 1)) + np.log(p0)

    pred = (log1 >= log0).astype(int)
    delta = np.clip(log1 - log0, -50, 50)
    prob1 = 1.0 / (1.0 + np.exp(-delta))

    return pred, prob1, bw1, bw0


## 15. 10-fold stratified CV for each feature

Rao builds **one separate Bayesian classifier per significant feature**.

Thus:
- Global → one classifier for each of the 46 significant global features;
- Local → one classifier for each of the 62 significant local activation features.

The reported comparison is the mean CV accuracy per feature.


In [ ]:
def evaluate_featurewise_bayes(df, features, feature_source, fold_df):
    work = df.copy()
    work["patient_id"] = work["patient_id"].astype(str)

    if len(work) != 577 or work["patient_id"].nunique() != 577:
        raise ValueError("Feature table must contain exactly 577 unique patients.")

    work = work.merge(
        fold_df[["patient_id", "fold"]],
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
    if work["fold"].isna().any():
        raise ValueError("Some patients do not have a shared fold assignment.")

    all_rows = []

    for feature in features:
        if feature not in work.columns:
            raise KeyError(f"Missing feature: {feature}")

        vals = work[feature].to_numpy(dtype=float)
        if not np.isfinite(vals).all():
            bad = work.loc[~np.isfinite(vals), "patient_id"].tolist()
            raise ValueError(
                f"{feature} contains non-finite values; cohort reduction is forbidden. "
                f"Patients: {bad[:10]}"
            )

        for fold in range(1, N_SPLITS + 1):
            train = work[work["fold"] != fold]
            valid = work[work["fold"] == fold]

            pred, prob, bw1, bw0 = bayes_kde_predict(
                train[feature].to_numpy(float),
                train["mgmt"].to_numpy(int),
                valid[feature].to_numpy(float),
            )
            yv = valid["mgmt"].to_numpy(int)

            all_rows.append({
                "source": feature_source,
                "feature": feature,
                "fold": fold,
                "accuracy": accuracy_score(yv, pred),
                "precision": precision_score(yv, pred, zero_division=0),
                "recall": recall_score(yv, pred, zero_division=0),
                "f1": f1_score(yv, pred, zero_division=0),
                "roc_auc": roc_auc_score(yv, prob),
                "bandwidth_mgmt1": bw1,
                "bandwidth_mgmt0": bw0,
            })

    fold_results = pd.DataFrame(all_rows)
    summary = (
        fold_results
        .groupby(["source", "feature"])
        .agg(
            mean_accuracy=("accuracy", "mean"),
            sd_accuracy=("accuracy", "std"),
            mean_precision=("precision", "mean"),
            mean_recall=("recall", "mean"),
            mean_f1=("f1", "mean"),
            mean_roc_auc=("roc_auc", "mean"),
        )
        .reset_index()
        .sort_values("mean_accuracy", ascending=False)
    )
    return fold_results, summary


## 16. Run Global Bayesian classifiers

In [ ]:
global_fold_results, global_cv_summary = evaluate_featurewise_bayes(
    global_z, global_sig, "global", shared_folds
)
global_fold_results.to_csv(OUTDIR / "global_bayes_10fold_per_fold.csv", index=False)
global_cv_summary.to_csv(OUTDIR / "global_bayes_10fold_summary.csv", index=False)
display(global_cv_summary.head(20))


## 17. Run Local Bayesian classifiers

In [ ]:
local_fold_results, local_cv_summary = evaluate_featurewise_bayes(
    local_activation, local_sig, "local", shared_folds
)
local_fold_results.to_csv(OUTDIR / "local_bayes_10fold_per_fold.csv", index=False)
local_cv_summary.to_csv(OUTDIR / "local_bayes_10fold_summary.csv", index=False)
display(local_cv_summary.head(20))


## 18. Compare features significant in both Global and Local approaches

The paper's Figure 11 compares common significant features and reports the percentage change in mean accuracy from Global to Local:

\[
\%\Delta Accuracy =
\frac{Acc_{local}-Acc_{global}}{Acc_{global}}\times100
\]


In [ ]:
g = global_cv_summary[
    ["feature", "mean_accuracy"]
].rename(
    columns={"mean_accuracy": "global_mean_accuracy"}
)

l = local_cv_summary[
    ["feature", "mean_accuracy"]
].rename(
    columns={"mean_accuracy": "local_mean_accuracy"}
)

comparison = g.merge(
    l,
    on="feature",
    how="inner",
)

comparison["percent_change_global_to_local"] = (
    (
        comparison["local_mean_accuracy"]
        - comparison["global_mean_accuracy"]
    )
    / comparison["global_mean_accuracy"]
    * 100.0
)

comparison = comparison.sort_values(
    "percent_change_global_to_local",
    ascending=False,
)

comparison.to_csv(
    OUTDIR / "global_vs_local_common_features.csv",
    index=False,
)

display(comparison)


## 19. Reproduce the paper-style result counts

In [ ]:
paper_check = pd.DataFrame({
    "quantity": [
        "patients",
        "MGMT methylated",
        "MGMT unmethylated",
        "global features extracted",
        "global significant p<0.05",
        "local features extracted",
        "local significant p<0.05",
        "common significant features",
    ],
    "observed": [
        len(manifest),
        int((manifest["mgmt"] == 1).sum()),
        int((manifest["mgmt"] == 0).sum()),
        len([c for c in global_raw.columns if c not in {"patient_id", "mgmt"}]),
        len(global_sig),
        len([c for c in local_activation.columns if c not in {"patient_id", "mgmt"}]),
        len(local_sig),
        len(comparison),
    ],
    "paper_reference": [
        577,
        301,
        276,
        93,
        46,
        93,
        62,
        "see paper Figure/Table",
    ],
})

display(paper_check)

paper_check.to_csv(
    OUTDIR / "paper_reproduction_check.csv",
    index=False,
)


## 20. Optional plots similar to Figures 9–11


In [ ]:
plt.figure(figsize=(12, 5))
tmp = global_cv_summary.sort_values("mean_accuracy", ascending=False)
plt.bar(range(len(tmp)), tmp["mean_accuracy"])
plt.xlabel("Significant global radiomic feature")
plt.ylabel("Mean 10-fold CV accuracy")
plt.title("Global Radiomic Bayesian Classifiers")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
tmp = local_cv_summary.sort_values("mean_accuracy", ascending=False)
plt.bar(range(len(tmp)), tmp["mean_accuracy"])
plt.xlabel("Significant local radiomic feature")
plt.ylabel("Mean 10-fold CV accuracy")
plt.title("Local Radiomic Bayesian Classifiers")
plt.tight_layout()
plt.show()

if len(comparison):
    plt.figure(figsize=(12, 5))
    tmp = comparison.sort_values(
        "percent_change_global_to_local",
        ascending=False,
    )
    plt.bar(
        range(len(tmp)),
        tmp["percent_change_global_to_local"],
    )
    plt.xlabel("Feature significant in both approaches")
    plt.ylabel("% change in mean accuracy")
    plt.title("Global → Local Bayesian Accuracy Change")
    plt.tight_layout()
    plt.show()


## 21. Outputs and methodological notes

Important outputs:

- `global_93_raw.csv`
- `global_ttest_results.csv`
- `local_93_percent_activation.csv`
- `local_ttest_results.csv`
- `rao_feature_name_match.json`
- `global_bayes_10fold_per_fold.csv`
- `local_bayes_10fold_per_fold.csv`
- shared folds: `shared_rao_comparison/mgmt_577_fold_assignment_10fold.csv`

Published checkpoints:
- 577 patients
- 301 MGMT methylated / 276 unmethylated
- 93 global features
- 46 significant global features
- 93 local features
- 62 significant local features

The notebook checks both the **counts and exact feature names** from Rao Tables 1 and 2.

The exact low-level PyRadiomics settings, KDE bandwidth grid, inner-CV count, t-test variance assumption and resize interpolation are not fully specified in the paper. They are therefore explicit implementation assumptions rather than claimed paper settings.

For reproduction, the cohort-level normalization/statistical-selection sequence follows Rao. A stricter leakage-controlled model-development analysis would move label-informed preprocessing inside each training fold.
